In [41]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

def load_vectorstore():
    # Define embeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    
    # Create sample documents for testing
    documents = [
        "We offer free shipping on orders over $50.",
        "You can track your order in My Orders section.",
        "Returns are accepted within 30 days of purchase."
    ]
    
    return FAISS.from_texts(texts=documents, embedding=embeddings)


In [46]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

def load_llm():
    pipe = pipeline(
        "text2text-generation",
        model="google/flan-t5-base",
        max_new_tokens=256,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.1,
        num_beams=2,
        min_length=20,
        no_repeat_ngram_size=3
    )
    llm = HuggingFacePipeline(pipeline=pipe)
    return llm  # Make sure you return this!


In [47]:
llm = load_llm()
print(llm)  # Should print HuggingFacePipeline object, not None
print(type(llm))  # Should be <class 'langchain_community.llms.huggingface_pipeline.HuggingFacePipeline'>


HuggingFacePipeline
Params: {'model_id': 'gpt2', 'model_kwargs': None, 'pipeline_kwargs': None}
<class 'langchain_community.llms.huggingface_pipeline.HuggingFacePipeline'>


In [58]:
test_response = llm(" where can i find my order details?")
print(test_response)  # Should generate some text


c:\Users\rajjd\miniconda3\envs\ecommerce-genai\lib\site-packages\transformers\generation\configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.3` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\rajjd\miniconda3\envs\ecommerce-genai\lib\site-packages\transformers\generation\configuration_utils.py:415: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


the website of the manufacturer of the product. i can find my order details on the website.


In [49]:
def load_llm():
    try:
        pipe = pipeline(
            "text2text-generation",
            model="google/flan-t5-base",
            device=-1,  # Force CPU (-1), or device=0 for GPU
            max_new_tokens=256
        )
        llm = HuggingFacePipeline(pipeline=pipe)
        return llm
    except Exception as e:
        print(f"Error loading LLM: {e}")
        return None


In [50]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

# Define your QA prompt
QA_PROMPT = PromptTemplate(
    template="Use the following context to answer the question.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:",
    input_variables=["context", "question"]
)

def create_agent(llm, vectorstore):
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True,
        output_key="answer"
    )
    
    qa_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
        memory=memory,
        combine_docs_chain_kwargs={"prompt": QA_PROMPT},
        return_source_documents=False
    )
    
    return qa_chain  # Return the chain, not result


In [52]:
from typing import Optional
import re

# Mock backend tools (same as project)
def get_order_status(order_id):
    return f"Order {order_id} is currently Shipped and will be delivered soon."

def create_return_request(order_id, reason):
    return f"Return request created for {order_id}. Reason: {reason}"

def get_refund_policy():
    return "Refunds are processed within 5–7 business days after pickup."

def payment_failed_help():
    return "Payment failed but deducted amounts are reversed in 5–7 business days."

def double_charge_help():
    return "Duplicate charges are usually reversed automatically within 5–7 days."


In [53]:
ECOMMERCE_KEYWORDS = [
    "order", "refund", "payment", "pay", "card", "upi",
    "shipping", "delivery", "track",
    "discount", "offer",
    "return", "replace", "exchange", "cancel",
]

def is_ecommerce_query(query: str) -> bool:
    q = query.lower()
    return any(word in q for word in ECOMMERCE_KEYWORDS)


In [54]:
def extract_order_id(text: str) -> Optional[str]:
    tokens = text.replace("#", " ").replace(",", " ").split()
    for t in tokens:
        t_clean = t.strip().upper().rstrip("?.!:,")
        if t_clean == "ORDER":
            continue
        if t_clean.startswith("ORD") and len(t_clean) > 3:
            return t_clean
        if t_clean.isdigit():
            return t_clean
    return None


In [55]:
def clean_answer(text: str) -> str:
    if not text:
        return ""
    patterns = [
        r"(?i)question:\s*",
        r"(?i)answer:\s*",
        r"(?i)assistant answer:.*",
    ]
    cleaned = text
    for p in patterns:
        cleaned = re.sub(p, "", cleaned)
    return re.sub(r"\s+", " ", cleaned).strip()


In [56]:
def agent(query: str) -> str:
    q_lower = query.lower()

    # Friendly thanks
    if any(x in q_lower for x in ["thank", "thanks", "appreciate"]):
        return "You’re welcome! Feel free to ask more shopping questions."

    # Out of scope
    if not is_ecommerce_query(query):
        return "I can help with orders, payments, returns, shipping, and offers."

    # Order tracking
    if "where is my order" in q_lower or "track" in q_lower:
        order_id = extract_order_id(query)
        if order_id:
            return get_order_status(order_id)
        return "Please provide your order ID (e.g., ORD123)."

    # Return / Exchange
    if "return" in q_lower or "exchange" in q_lower:
        order_id = extract_order_id(query)
        if not order_id:
            return "Please include order ID to raise a return."
        return create_return_request(order_id, query)

    # Refund policy
    if "refund policy" in q_lower:
        return get_refund_policy()

    # Payment issues
    if "payment failed" in q_lower or "money deducted" in q_lower:
        return payment_failed_help()

    # KB / LLM fallback (simulated)
    return "Delivery usually takes 3–7 business days depending on location."


In [60]:
tool_tests = [
    "Can you teach me dance?",
    "Track my order ORD999",
    "I want to return order ORD789 because it is damaged",
    "Replace order ORD456 with another size",
    "What is your refund policy?",
    "My payment failed but money was deducted"
]

for q in tool_tests:
    print("User:", q)
    print("Bot :", agent(q))
    print("-" * 50)


User: Can you teach me dance?
Bot : I can help with orders, payments, returns, shipping, and offers.
--------------------------------------------------
User: Track my order ORD999
Bot : Order ORD999 is currently Shipped and will be delivered soon.
--------------------------------------------------
User: I want to return order ORD789 because it is damaged
Bot : Return request created for ORD789. Reason: I want to return order ORD789 because it is damaged
--------------------------------------------------
User: Replace order ORD456 with another size
Bot : Delivery usually takes 3–7 business days depending on location.
--------------------------------------------------
User: What is your refund policy?
Bot : Refunds are processed within 5–7 business days after pickup.
--------------------------------------------------
User: My payment failed but money was deducted
Bot : Payment failed but deducted amounts are reversed in 5–7 business days.
-------------------------------------------------

In [62]:
edge_cases = [
    "where is my order",
    "Who is the Prime Minister of India?",
    "Hello",
    "Thanks a lot"
]

for q in edge_cases:
    print("User:", q)
    print("Bot :", agent(q))
    print("-" * 50)


User: where is my order
Bot : Please provide your order ID (e.g., ORD123).
--------------------------------------------------
User: Who is the Prime Minister of India?
Bot : I can help with orders, payments, returns, shipping, and offers.
--------------------------------------------------
User: Hello
Bot : I can help with orders, payments, returns, shipping, and offers.
--------------------------------------------------
User: Thanks a lot
Bot : You’re welcome! Feel free to ask more shopping questions.
--------------------------------------------------
